# 27 - Embedding cosine vs chemical similarity, three fingerprint types

`24_embedding_vs_tanimoto_unique_pairs.ipynb` reports a single
Spearman correlation - embedding cosine vs **ECFP4** Tanimoto - over
unique ligand-code pairs, and reads the moderate rho as evidence that
DrugCLIP ligand embeddings are not simply recapitulating conventional
fingerprint-based chemical similarity. That raises an obvious
question: is the weak correlation specific to ECFP4 (a hashed
circular-substructure fingerprint), or does it hold for other, very
differently constructed similarity measures too?

This notebook reuses notebook 24's exact unique-pair machinery
(cross-dataset dedup, exact mean-cosine-per-pair via one matmul, self
vs distinct-code views) and adds two more chemical-similarity axes
alongside ECFP4 Tanimoto:

1. **MACCS keys Tanimoto** - 166 predefined substructure keys, a much
   coarser and differently-constructed fingerprint than a 2048-bit
   hashed circular one.
2. **Physicochemical descriptor similarity** - 10 global RDKit
   descriptors (MW, logP, TPSA, H-bond donor/acceptor counts,
   rotatable bonds, aromatic ring count, Csp3 fraction, ring count,
   heavy-atom count), z-scored per scope, similarity =
   `1 / (1 + Euclidean distance)`. This has no notion of substructure
   at all - two molecules can score high here while sharing no
   scaffold, or low while sharing most of one.

Everything below is done **per individual dataset and on the pooled,
cross-dataset-deduplicated combination of all four**, average
embedding only (folds behave the same throughout this project's other
notebooks, so are not re-litigated here).

Regenerate via `scripts/build_embedding_vs_multifp_notebook.py`.

## Setup

In [ ]:
import time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, spearmanr

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

PROJECT = Path.cwd().parent / 'DATA' / 'natural_ligands'
MOLS_DIR = PROJECT / 'RESULTS' / 'mols'
CCD_MANIFEST = (PROJECT / 'RESULTS' /
                'components_pub_unimol_manifest.tsv')
OUT_DIR = PROJECT / 'RESULTS' / 'embedding_vs_multifp'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ['coach420', 'scpdb', 'pdbbind2020', 'holo4k']
N_FOLDS = 6
EMB_DIM = 128

FP_RADIUS, FP_BITS = 2, 2048   # ECFP4
DESC_NAMES = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
              'NumRotatableBonds', 'NumAromaticRings', 'FractionCSP3',
              'RingCount', 'HeavyAtomCount']
FP_TYPES = ['ecfp4', 'maccs', 'physchem']

# Axis-label / title / tick font sizes for the plots below - matches
# notebook 24's configurable font sizes.
LABEL_FONTSIZE = 13     # ax.set_xlabel / ax.set_ylabel
TITLE_FONTSIZE = 13     # ax.set_title
SUPTITLE_FONTSIZE = 13  # fig.suptitle
TICK_FONTSIZE = 13      # plain numeric tick labels (ax.tick_params)
CAT_TICK_FONTSIZE = 13  # rotated categorical tick labels

print('descriptor set:', DESC_NAMES)

## Load embeddings + all three fingerprint/descriptor representations

Same loader as notebook 24 (including the lexicographic lmdb-key
re-sort needed to align `mol_reps` rows to the manifest), extended to
also build a MACCS bit vector and a raw physchem descriptor vector per
ligand code (on top of the ECFP4 bit vector already used in nb24). A
code is dropped only if its SMILES fails to parse at all - not if one
particular descriptor fails, since all three representations use the
same RDKit `Mol` object.

In [ ]:
ccd = pd.read_csv(CCD_MANIFEST, sep='\t', dtype=str)
code_to_smi = dict(zip(ccd['identifier'], ccd['smiles']))

_rep_cache = {}
def reps_for_code(c):
    '''code -> dict(ecfp4=bitvect, maccs=bitvect, desc=np.ndarray) or None.'''
    if c in _rep_cache:
        return _rep_cache[c]
    smi = code_to_smi.get(c, '')
    m = Chem.MolFromSmiles(smi) if smi else None
    if m is None:
        _rep_cache[c] = None
        return None
    rep = {
        'ecfp4': AllChem.GetMorganFingerprintAsBitVect(m, FP_RADIUS, FP_BITS),
        'maccs': MACCSkeys.GenMACCSKeys(m),
        'desc': np.array([getattr(Descriptors, n)(m) for n in DESC_NAMES],
                         dtype=np.float64),
    }
    _rep_cache[c] = rep
    return rep

def load_dataset(ds):
    with h5py.File(MOLS_DIR / f'bound_{ds}.h5', 'r') as f:
        reps = f['mol_reps'][:]                       # (N, 768)
    n = reps.shape[0]
    emb = reps.reshape(n, N_FOLDS, EMB_DIM).astype(np.float32)
    man = pd.read_csv(MOLS_DIR / f'bound_{ds}_manifest.tsv',
                      sep='\t', dtype=str)
    if len(man) != n:
        raise ValueError(f'{ds}: manifest {len(man)} != h5 {n} rows')
    # h5 rows are in lexicographic lmdb-key order, not numeric; re-sort
    # the manifest to match so emb[i] <-> codes[i] (see notebook 18/24).
    man = man.sort_values(
        'lmdb_key', key=lambda s: s.astype(str)).reset_index(drop=True)
    codes = man['lig_resname'].fillna('').to_numpy()
    pdb_ids = man['pdb_id'].fillna('').to_numpy()
    reps_list = [reps_for_code(c) for c in codes]
    keep = np.array([r is not None for r in reps_list])
    return {'emb': emb[keep], 'codes': codes[keep],
            'pdb_id': pdb_ids[keep],
            'reps': [r for r, k in zip(reps_list, keep) if k],
            'n_total': n, 'n_kept': int(keep.sum())}

data = {}
rows = []
for ds in DATASETS:
    d = load_dataset(ds)
    data[ds] = d
    rows.append({'dataset': ds, 'n_mols': d['n_total'],
                 'with_reps': d['n_kept'],
                 'coverage': d['n_kept'] / d['n_total'],
                 'unique_codes': len(set(d['codes']))})
cover = pd.DataFrame(rows)
cover['coverage'] = (cover['coverage'] * 100).round(1)
print(cover.to_string(index=False))

## Pool datasets + cross-dataset dedup

Identical to notebook 24: for every `(pdb_id, lig_resname)` appearing
in more than one dataset, keep only one representative (first dataset
in `DATASETS` order) before pooling, so the same physical pose is not
double- or triple-counted. Repeats **within** one dataset are kept -
they are distinct binding sites.

In [ ]:
frames = []
for ds in DATASETS:
    d = data[ds]
    frames.append(pd.DataFrame({'dataset': ds, 'pdb_id': d['pdb_id'],
                                'code': d['codes'],
                                'row': np.arange(len(d['codes']))}))
pooled_man = pd.concat(frames, ignore_index=True)

priority = {ds: i for i, ds in enumerate(DATASETS)}
pooled_man['prio'] = pooled_man['dataset'].map(priority)
chosen = pooled_man.groupby(['pdb_id', 'code'])['prio'].transform('min')
keep_mask = (pooled_man['prio'] == chosen).to_numpy()

emb_pooled_all = np.concatenate([data[ds]['emb'] for ds in DATASETS], axis=0)
reps_pooled_all = sum((data[ds]['reps'] for ds in DATASETS), [])
codes_pooled_all = pooled_man['code'].to_numpy()

emb_pooled = emb_pooled_all[keep_mask]
codes_pooled = codes_pooled_all[keep_mask]
reps_pooled = [r for r, k in zip(reps_pooled_all, keep_mask) if k]

print(f'pooled: {len(pooled_man):,} rows -> {keep_mask.sum():,} after '
      f'cross-dataset dedup ({(~keep_mask).sum():,} dropped, '
      f'{len(set(codes_pooled)):,} unique codes)')

SCOPES = {ds: {'codes': data[ds]['codes'], 'reps': data[ds]['reps'],
              'emb': data[ds]['emb']} for ds in DATASETS}
SCOPES['pooled'] = {'codes': codes_pooled, 'reps': reps_pooled,
                    'emb': emb_pooled}
print('\nscopes:', {k: len(v['codes']) for k, v in SCOPES.items()})

## Helper functions

- `full_tanimoto_matrix` - exact pairwise Tanimoto over unique ligand
  codes, for either ECFP4 or MACCS bit vectors (same function, any
  RDKit `ExplicitBitVect`).
- `full_physchem_sim_matrix` - z-scores the 10-d descriptor vector
  across the scope's unique codes, then converts pairwise Euclidean
  distance to a similarity via `1 / (1 + distance)`.
- `unique_pair_table` - the nb24 core, generalised to carry all three
  similarity columns per unique ligand-code pair instead of one.

In [ ]:
def unit_rows(M):
    nrm = np.linalg.norm(M, axis=1, keepdims=True)
    return M / np.clip(nrm, 1e-12, None)

def full_tanimoto_matrix(fps):
    n = len(fps)
    mat = np.ones((n, n), dtype=np.float32)
    for i in range(n - 1):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:])
        mat[i, i + 1:] = sims
        mat[i + 1:, i] = sims
    return mat

def full_physchem_sim_matrix(desc):
    z = (desc - desc.mean(0)) / np.clip(desc.std(0), 1e-12, None)
    d = cdist(z, z, metric='euclidean')
    return (1.0 / (1.0 + d)).astype(np.float32)

def factorize_codes(codes):
    code_idx, uniq_codes = pd.factorize(codes, sort=True)
    return code_idx.astype(np.int32), np.asarray(uniq_codes)

def unique_pair_table(code_idx, n_codes, emb, sim_mats, uniq_codes):
    E = unit_rows(emb.mean(axis=1))
    cos_full = (E @ E.T).astype(np.float32)
    iu = np.triu_indices(E.shape[0], k=1)
    cosv = cos_full[iu]
    del cos_full
    ci, cj = code_idx[iu[0]], code_idx[iu[1]]
    lo = np.minimum(ci, cj); hi = np.maximum(ci, cj)
    key = lo.astype(np.int64) * n_codes + hi.astype(np.int64)
    g = pd.DataFrame({'key': key, 'cos': cosv}).groupby('key')['cos'].agg(
        ['mean', 'std', 'count'])
    lo_g = (g.index // n_codes).to_numpy()
    hi_g = (g.index % n_codes).to_numpy()
    n_g = g['count'].to_numpy()
    out = pd.DataFrame({
        'code_a': uniq_codes[lo_g], 'code_b': uniq_codes[hi_g],
        'is_self': lo_g == hi_g,
        'n_instance_pairs': n_g,
        'mean_cos': g['mean'].to_numpy(),
        'sem_cos': (g['std'] / np.sqrt(n_g)).to_numpy(),
    })
    for name, mat in sim_mats.items():
        out[name] = mat[lo_g, hi_g]
    return out

def corr_stats(tab, fp_col):
    r = pearsonr(tab[fp_col], tab['mean_cos']).statistic
    rho = spearmanr(tab[fp_col], tab['mean_cos']).statistic
    dist = tab[~tab['is_self']]
    if len(dist) > 10:
        r_d = pearsonr(dist[fp_col], dist['mean_cos']).statistic
        rho_d = spearmanr(dist[fp_col], dist['mean_cos']).statistic
    else:
        r_d = rho_d = np.nan
    return r, rho, r_d, rho_d

## Compute unique-pair tables (per scope, all three similarity types)

For each scope: factorize ligand codes, build all three full
similarity matrices over unique codes once (ECFP4 Tanimoto, MACCS
Tanimoto, physchem similarity - all mode-independent), then the
average-embedding unique-pair table joining all three plus mean
cosine. Correlation stats (Pearson/Spearman, all-pairs and
distinct-code-only) are computed per fingerprint type.

In [ ]:
res_rows = []
pair_tables = {}
for scope, s in SCOPES.items():
    t0 = time.time()
    code_idx, uniq_codes = factorize_codes(s['codes'])
    n_codes = len(uniq_codes)
    code_to_rep = {}
    for c, r in zip(s['codes'], s['reps']):
        code_to_rep.setdefault(c, r)
    uniq_reps = [code_to_rep[c] for c in uniq_codes]
    sim_mats = {
        'ecfp4': full_tanimoto_matrix([r['ecfp4'] for r in uniq_reps]),
        'maccs': full_tanimoto_matrix([r['maccs'] for r in uniq_reps]),
        'physchem': full_physchem_sim_matrix(
            np.stack([r['desc'] for r in uniq_reps])),
    }
    tab = unique_pair_table(code_idx, n_codes, s['emb'], sim_mats, uniq_codes)
    pair_tables[scope] = tab
    for fp in FP_TYPES:
        r, rho, r_d, rho_d = corr_stats(tab, fp)
        res_rows.append({'scope': scope, 'fp_type': fp,
                         'n_unique_pairs': len(tab),
                         'n_self': int(tab['is_self'].sum()),
                         'pearson': r, 'spearman': rho,
                         'pearson_distinct': r_d,
                         'spearman_distinct': rho_d})
    print(f'{scope:12s} n_codes={n_codes:5d} '
          f'unique_pairs={len(tab):,}  ({time.time()-t0:.1f}s)')

res_multifp = pd.DataFrame(res_rows)
res_multifp.to_csv(OUT_DIR / 'multifp_correlations.tsv', sep='\t',
                   index=False)
print('\nwrote', OUT_DIR / 'multifp_correlations.tsv')

# Cache the per-scope unique-pair tables themselves (not just the
# correlation summary) - these are what every plot below is built
# from, and rebuilding them (the full similarity-matrix computation)
# is the slow part of this notebook. Caching them lets plot-only
# tweaks be iterated on without re-running the whole notebook.
import pickle
with open(OUT_DIR / 'pair_tables.pkl', 'wb') as f:
    pickle.dump(pair_tables, f)
print('wrote', OUT_DIR / 'pair_tables.pkl', '(cache for fast plot iteration)')
res_multifp.round(3)

## Per-scope plots: binned median + IQR trend per fingerprint type

For each scope: the same trend formatting as notebook 24's binned
median + IQR plot (`plot_scope_trend`) - every distinct-code unique
pair as a faint background scatter point (marker size ~ log10
supporting instance-pairs, grey, alpha 0.05), overlaid with the
per-bin median (skyblue line) and IQR band (skyblue fill) across 40
similarity bins - repeated for each of the three similarity types,
plus a bar panel comparing Spearman rho across the three (all-pairs
and distinct-code-only side by side).

In [ ]:
FP_LABELS = {'ecfp4': 'ECFP4 Tanimoto', 'maccs': 'MACCS Tanimoto',
            'physchem': 'Physicochemical similarity'}

def plot_scope_multifp(scope, n_bins=40):
    tab = pair_tables[scope]
    sub = res_multifp[res_multifp['scope'] == scope].set_index('fp_type')
    dist = tab[~tab['is_self']]
    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    sizes = 3 + 5 * np.log10(dist['n_instance_pairs'])
    fig, axes = plt.subplots(1, 4, figsize=(23, 5.5))
    for ax, fp in zip(axes[:3], FP_TYPES):
        bin_idx = np.clip(np.digitize(dist[fp], edges) - 1, 0, n_bins - 1)
        stats = dist['mean_cos'].groupby(bin_idx).agg(
            median='median', q25=lambda s: s.quantile(0.25),
            q75=lambda s: s.quantile(0.75), n='size')
        stats = stats.reindex(range(n_bins))
        ax.scatter(dist[fp], dist['mean_cos'], s=sizes, color='0.6',
                  alpha=0.05, edgecolors='none', rasterized=True, zorder=1)
        ax.plot(centers, stats['median'], color='skyblue', marker='o', ms=4,
               label='Median', zorder=3)
        ax.fill_between(centers, stats['q25'], stats['q75'], color='skyblue',
                        alpha=0.2, label='IQR', zorder=2)
        rho_d = sub.loc[fp, 'spearman_distinct']
        ax.set_xlabel(FP_LABELS[fp], fontsize=LABEL_FONTSIZE)
        ax.set_ylabel('mean embedding cosine', fontsize=LABEL_FONTSIZE)
        ax.set_xlim(0, 1); ax.set_ylim(-1, 1)
        ax.set_xticks(np.arange(0, 1.01, 0.1))
        ax.set_box_aspect(1)
        ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
        ax.set_title(f'{FP_LABELS[fp]}\nrho={rho_d:.2f} (distinct-code pairs)',
                    fontsize=TITLE_FONTSIZE)
        ax.legend(fontsize=8, loc='lower right')
    ax = axes[3]
    x = np.arange(len(FP_TYPES))
    ax.bar(x - 0.2, [sub.loc[fp, 'spearman'] for fp in FP_TYPES], width=0.4,
          label='all pairs', color='#4c72b0')
    ax.bar(x + 0.2, [sub.loc[fp, 'spearman_distinct'] for fp in FP_TYPES],
          width=0.4, label='distinct only', color='#c44e52')
    ax.set_xticks(x); ax.set_xticklabels([FP_LABELS[fp] for fp in FP_TYPES],
                                        rotation=20, ha='right',
                                        fontsize=CAT_TICK_FONTSIZE)
    ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
    ax.set_ylabel('Spearman rho', fontsize=LABEL_FONTSIZE)
    ax.set_title('rho by fingerprint type', fontsize=TITLE_FONTSIZE)
    ax.axhline(0, color='k', lw=0.6); ax.legend(fontsize=8)
    fig.suptitle(f'{scope}: embedding cosine vs three similarity types',
                fontsize=SUPTITLE_FONTSIZE)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    out = OUT_DIR / f'fig_{scope}_multifp.png'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    print('wrote', out)
    plt.show()

for scope in SCOPES:
    plot_scope_multifp(scope)

## Pooled scope: individual trend plots per fingerprint type

Manuscript-style standalone figure per fingerprint type (pooled scope
only, distinct-code pairs only) - matching the existing
`N = .. / rho = ..` annotation-box convention, with a pastel
colour per fingerprint type (tan / orchid / sky blue, in the same
family as the ChimeraX default per-model palette) in place of a
single shared `skyblue`.

In [ ]:
FP_COLORS = {'ecfp4': '#D2B48C', 'maccs': '#87CEEB', 'physchem': '#DDA0DD'}

def plot_pooled_individual(fp, n_bins=40):
    tab = pair_tables['pooled']
    sub = res_multifp[(res_multifp['scope'] == 'pooled') &
                      (res_multifp['fp_type'] == fp)].iloc[0]
    dist = tab[~tab['is_self']]
    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    bin_idx = np.clip(np.digitize(dist[fp], edges) - 1, 0, n_bins - 1)
    stats = dist['mean_cos'].groupby(bin_idx).agg(
        median='median', q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75), n='size')
    stats = stats.reindex(range(n_bins))
    sizes = 3 + 5 * np.log10(dist['n_instance_pairs'])
    color = FP_COLORS[fp]
    n_pts = len(dist)
    rho = sub['spearman_distinct']

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(dist[fp], dist['mean_cos'], s=sizes, color='0.6',
              alpha=0.05, edgecolors='none', rasterized=True, zorder=1)
    ax.plot(centers, stats['median'], color=color, marker='o', ms=4,
           label='Median', zorder=3)
    ax.fill_between(centers, stats['q25'], stats['q75'], color=color,
                    alpha=0.35, label='IQR', zorder=2)
    ax.set_xlabel(FP_LABELS[fp], fontsize=LABEL_FONTSIZE)
    ax.set_ylabel('Embedding cosine similarity', fontsize=LABEL_FONTSIZE)
    ax.set_xlim(0, 1); ax.set_ylim(-1, 1)
    ax.set_xticks(np.arange(0, 1.01, 0.1))
    ax.set_box_aspect(1)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.text(0.97, 0.05, f'N = {n_pts:,}\n' + r'$\rho$' + f' = {rho:.2f}',
           transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='0.5',
                    alpha=0.9))
    fig.tight_layout()
    out = OUT_DIR / f'fig_pooled_{fp}_trend.png'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    print(f'{fp:10s} N={n_pts:,}  rho={rho:.3f}  wrote {out}')
    plt.show()

for fp in FP_TYPES:
    plot_pooled_individual(fp)

## Summary heatmap: Spearman rho, scope x fingerprint type

In [ ]:
pivot_all = res_multifp.pivot(index='scope', columns='fp_type',
                              values='spearman').reindex(
    index=list(SCOPES), columns=FP_TYPES)
pivot_dist = res_multifp.pivot(index='scope', columns='fp_type',
                               values='spearman_distinct').reindex(
    index=list(SCOPES), columns=FP_TYPES)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, piv, title in [(ax0, pivot_all, 'all pairs (incl. self)'),
                       (ax1, pivot_dist, 'distinct-code pairs only')]:
    im = ax.imshow(piv.values, aspect='auto', cmap='magma',
                   vmin=np.nanmin(piv.values), vmax=np.nanmax(piv.values))
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([FP_LABELS[c] for c in piv.columns], rotation=20,
                      ha='right')
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    for yi in range(piv.shape[0]):
        for xi in range(piv.shape[1]):
            v = piv.values[yi, xi]
            if np.isfinite(v):
                ax.text(xi, yi, f'{v:.2f}', ha='center', va='center',
                       color='w', fontsize=9)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Spearman rho: embedding cosine vs chemical similarity, '
             'by fingerprint type', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
out = OUT_DIR / 'fig_multifp_heatmap.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
print('wrote', out)
plt.show()

## Takeaways

- Read the distinct-code-only heatmap panel first - self-pairs sit at
  similarity = 1 for all three fingerprint types by construction and
  can only inflate agreement.
- **If rho stays in the same (weak-to-moderate) range across all
  three fingerprint types**, the notebook-24/18 result is not an ECFP4
  artefact: embedding cosine fails to track chemical similarity
  however that similarity is defined (circular substructure keys,
  predefined substructure keys, or bulk physicochemical descriptors).
- **If rho diverges sharply by type** (e.g. tracks physchem similarity
  much better/worse than either fingerprint), that is itself an
  interesting, reportable result and changes what can be claimed about
  what the embedding does and does not encode - it should replace, not
  just supplement, a single-fingerprint correlation number in the
  manuscript.
- This notebook only speaks to *whether the weak correlation
  generalises across chemical-similarity definitions*. It says nothing
  about *why* embedding cosine behaves the way it does - for that,
  see notebook 28, which stratifies by pocket similarity instead.